# Quantum Radioactive Decay: Data Analysis and Machine Learning

This notebook implements the complete data-analysis workflow from the reference notebook in a simpler, efficient structure:

1. Load and inspect the radioactive decay dataset.
2. Audit missing values, clean the data, encode categories, and check outliers.
3. Explore decay curves, experiment groups, distributions, and normality.
4. Study Pearson and Spearman correlations with the prediction target.
5. Create physics-inspired features from the exponential decay law.
6. Train and compare regression models using R2, MAE, and RMSE.
7. Diagnose predictions with actual-vs-predicted plots, residuals, feature importance, and cross-validation.


In [ ]:
# 1. Import libraries and set global options

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

try:
    display
except NameError:
    display = print

RANDOM_STATE = 42
DATA_PATH = Path("quantum_radioactive_decay_dataset.csv")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

print("Libraries loaded successfully.")


In [ ]:
# 2. Load the dataset

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())
display(df.tail())


In [ ]:
# 3. Basic exploration

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})

print("Columns:")
print(list(df.columns))

display(overview)
display(df.describe(include="all").T)


In [ ]:
# 4. Missing value audit

missing = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent,
}).sort_values("missing_count", ascending=False)

display(missing_report[missing_report["missing_count"] > 0])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

missing_report["missing_count"].plot(kind="bar", ax=axes[0], color="#2a9d8f")
axes[0].set_title("Missing Values by Column")
axes[0].set_ylabel("Missing Count")
axes[0].tick_params(axis="x", rotation=45)

sample_missing = df.sample(min(800, len(df)), random_state=RANDOM_STATE).isna().T
sns.heatmap(sample_missing, cbar=False, ax=axes[1], cmap=["#f8f9fa", "#e76f51"])
axes[1].set_title("Missing Pattern in Sampled Rows")
axes[1].set_xlabel("Sampled rows")
axes[1].set_ylabel("Columns")

plt.tight_layout()
plt.show()


In [ ]:
# 5. Clean missing values, encode category, and audit outliers

df_clean = df.copy()

# These columns vary across each sample's time series, so interpolation keeps the
# local decay curve smoother than a global mean or median fill.
time_series_cols = ["time", "remaining_atoms", "decayed_atoms"]
for col in time_series_cols:
    df_clean[col] = (
        df_clean.groupby("sample_id")[col]
        .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )

# Decay constant is fixed within a sample, so fill from the same sample first.
df_clean["decay_constant"] = (
    df_clean.groupby("sample_id")["decay_constant"]
    .transform(lambda s: s.ffill().bfill())
)
df_clean["decay_constant"] = df_clean["decay_constant"].fillna(
    df_clean["decay_constant"].median()
)

# Final numeric fallback for any rare edge cases.
numeric_columns = df_clean.select_dtypes(include=np.number).columns
df_clean[numeric_columns] = df_clean[numeric_columns].fillna(df_clean[numeric_columns].median())

# Keep the key physics columns internally consistent after imputation.
df_clean["remaining_atoms"] = df_clean["remaining_atoms"].clip(lower=0)
df_clean["remaining_atoms"] = np.minimum(df_clean["remaining_atoms"], df_clean["initial_atoms"])
df_clean["decayed_atoms"] = (df_clean["initial_atoms"] - df_clean["remaining_atoms"]).clip(lower=0)
df_clean["probability_of_decay"] = (
    df_clean["decayed_atoms"] / df_clean["initial_atoms"]
).clip(0, 1)
df_clean["half_life"] = np.log(2) / df_clean["decay_constant"]

df_clean["experiment_group_code"] = df_clean["experiment_group"].astype("category").cat.codes

NUMERIC_COLS = [
    "initial_atoms",
    "decay_constant",
    "time",
    "remaining_atoms",
    "decayed_atoms",
    "probability_of_decay",
    "half_life",
    "noise_level",
]

def count_iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

outlier_report = pd.DataFrame({
    "outlier_count": {col: count_iqr_outliers(df_clean[col]) for col in NUMERIC_COLS}
}).sort_values("outlier_count", ascending=False)

cleaning_check = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "missing_after_cleaning": df_clean.isna().sum(),
})

print("Missing values after cleaning:", int(df_clean.isna().sum().sum()))
print("Rows retained:", f"{len(df_clean):,}")
display(cleaning_check)
display(outlier_report)



In [ ]:
# 6. Descriptive statistics with shape indicators

desc = df_clean[NUMERIC_COLS].describe().T
desc["skewness"] = df_clean[NUMERIC_COLS].skew()
desc["kurtosis"] = df_clean[NUMERIC_COLS].kurtosis()

display(desc.round(3))


In [ ]:
# 7. Visualize decay curves for a small set of samples

rng = np.random.default_rng(RANDOM_STATE)
sample_ids = rng.choice(
    df_clean["sample_id"].unique(),
    size=min(12, df_clean["sample_id"].nunique()),
    replace=False,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sample_id in sample_ids:
    sample = df_clean[df_clean["sample_id"] == sample_id].sort_values("time")
    axes[0].plot(sample["time"], sample["remaining_atoms"], alpha=0.75)
    axes[1].plot(sample["time"], sample["probability_of_decay"], alpha=0.75)

axes[0].set_title("Remaining Atoms Over Time")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Remaining Atoms")

axes[1].set_title("Decay Probability Over Time")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Probability of Decay")

plt.tight_layout()
plt.show()


In [ ]:
# 8. Experiment group analysis

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

group_counts = df_clean["experiment_group"].value_counts().sort_index()
sns.barplot(x=group_counts.index, y=group_counts.values, ax=axes[0])
axes[0].set_title("Rows per Experiment Group")
axes[0].set_xlabel("Experiment Group")
axes[0].set_ylabel("Row Count")

group_decay = df_clean.groupby("experiment_group")["decay_constant"].mean().sort_index()
sns.barplot(x=group_decay.index, y=group_decay.values, ax=axes[1])
axes[1].set_title("Mean Decay Constant by Group")
axes[1].set_xlabel("Experiment Group")
axes[1].set_ylabel("Mean Decay Constant")

sns.boxplot(data=df_clean, x="experiment_group", y="noise_level", ax=axes[2])
axes[2].set_title("Noise Level by Group")
axes[2].set_xlabel("Experiment Group")
axes[2].set_ylabel("Noise Level")

plt.tight_layout()
plt.show()


In [ ]:
# 9. Scatter matrix for key relationships

key_cols = ["time", "remaining_atoms", "decayed_atoms", "probability_of_decay", "half_life"]
scatter_sample = df_clean[key_cols].sample(min(2000, len(df_clean)), random_state=RANDOM_STATE)

axes = pd.plotting.scatter_matrix(
    scatter_sample,
    figsize=(12, 12),
    diagonal="hist",
    alpha=0.15,
    hist_kwds={"bins": 30},
)

for ax in axes.ravel():
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle("Scatter Matrix of Key Decay Variables", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 10. Distribution plots for numeric features

plot_sample = df_clean.sample(min(10000, len(df_clean)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.ravel()

for i, col in enumerate(NUMERIC_COLS):
    sns.histplot(plot_sample[col], bins=40, kde=True, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("")

axes[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Q-Q plots and normality checks

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

normality_rows = []
for i, col in enumerate(NUMERIC_COLS):
    sample = df_clean[col].sample(min(1000, len(df_clean)), random_state=RANDOM_STATE)
    stats.probplot(sample, dist="norm", plot=axes[i])
    axes[i].set_title(col)

    shapiro_sample = sample.iloc[:500]
    _, p_value = stats.shapiro(shapiro_sample)
    normality_rows.append({
        "feature": col,
        "shapiro_p_value": p_value,
        "normal_at_0_05": p_value > 0.05,
    })

plt.tight_layout()
plt.show()

display(pd.DataFrame(normality_rows).round(4))


In [ ]:
# 12. Violin and box plots by experiment group

group_sample = pd.concat([
    group.sample(min(500, len(group)), random_state=RANDOM_STATE)
    for _, group in df_clean.groupby("experiment_group")
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(data=group_sample, x="experiment_group", y="remaining_atoms", ax=axes[0])
axes[0].set_title("Remaining Atoms by Experiment Group")
axes[0].set_xlabel("Experiment Group")
axes[0].set_ylabel("Remaining Atoms")

sns.boxplot(data=df_clean, x="experiment_group", y="half_life", ax=axes[1])
axes[1].set_title("Half-Life by Experiment Group")
axes[1].set_xlabel("Experiment Group")
axes[1].set_ylabel("Half-Life")

plt.tight_layout()
plt.show()


In [ ]:
# 13. Correlation analysis: Pearson and Spearman

pearson_corr = df_clean[NUMERIC_COLS].corr(method="pearson")
spearman_corr = df_clean[NUMERIC_COLS].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, corr_matrix, title in [
    (axes[0], pearson_corr, "Pearson Correlation"),
    (axes[1], spearman_corr, "Spearman Correlation"),
]:
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()


In [ ]:
# 14. Features most correlated with the prediction target

TARGET = "remaining_atoms"
corr_target = (
    pearson_corr[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
)

display(corr_target.to_frame("pearson_correlation").round(4))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#2a9d8f" if value > 0 else "#e76f51" for value in corr_target.values]
ax.barh(corr_target.index, corr_target.values, color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title(f"Correlation with {TARGET}")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

print("Interpretation:")
for feature, value in corr_target.items():
    strength = "strong" if abs(value) >= 0.70 else "moderate" if abs(value) >= 0.40 else "weak"
    direction = "positive" if value > 0 else "negative"
    print(f"{feature:25s}: {strength:8s} {direction:8s} ({value:.3f})")


In [ ]:
# 15. Scatter plots for the strongest relationships

top_features = corr_target.head(4).index.tolist()
scatter_sample = df_clean.sample(min(3000, len(df_clean)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for i, feature in enumerate(top_features):
    sns.regplot(
        data=scatter_sample,
        x=feature,
        y=TARGET,
        scatter_kws={"alpha": 0.25, "s": 12},
        line_kws={"color": "#e76f51"},
        ax=axes[i],
    )
    axes[i].set_title(f"{feature} vs {TARGET}")

plt.tight_layout()
plt.show()


In [ ]:
# 16. Physics-inspired feature engineering

df_feat = df_clean.copy()

df_feat["theoretical_remaining"] = (
    df_feat["initial_atoms"] * np.exp(-df_feat["decay_constant"] * df_feat["time"])
)
df_feat["decay_residual"] = df_feat["remaining_atoms"] - df_feat["theoretical_remaining"]
df_feat["decay_rate"] = df_feat["decay_constant"] * df_feat["remaining_atoms"]
df_feat["time_in_half_lives"] = df_feat["time"] / df_feat["half_life"]
df_feat["log_remaining"] = np.log1p(df_feat["remaining_atoms"])
df_feat["decay_fraction"] = df_feat["decayed_atoms"] / df_feat["initial_atoms"]

ENGINEERED_COLS = [
    "theoretical_remaining",
    "decay_residual",
    "decay_rate",
    "time_in_half_lives",
    "log_remaining",
    "decay_fraction",
]

display(df_feat[ENGINEERED_COLS].describe().T.round(3))

feature_corr_cols = NUMERIC_COLS + ENGINEERED_COLS
engineered_corr = (
    df_feat[feature_corr_cols]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
)

display(engineered_corr.head(10).to_frame("abs_sorted_correlation").round(4))


## Modeling note

For prediction, target-derived columns such as `decayed_atoms`, `probability_of_decay`, `decay_residual`, `decay_rate`, `log_remaining`, and `decay_fraction` are useful for analysis, but they already contain information from `remaining_atoms`.

The model below uses only features that can be known before predicting the remaining atoms.


In [ ]:
# 17. Prepare model features and train/test split

model_df = pd.get_dummies(
    df_feat,
    columns=["experiment_group"],
    prefix="group",
    dtype=int,
)

group_features = [col for col in model_df.columns if col.startswith("group_")]

MODEL_FEATURES = [
    "initial_atoms",
    "decay_constant",
    "time",
    "half_life",
    "noise_level",
    "theoretical_remaining",
    "time_in_half_lives",
] + group_features

X = model_df[MODEL_FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows : {len(X_test):,}")
print(f"Features     : {len(MODEL_FEATURES)}")
display(pd.Series(MODEL_FEATURES, name="model_features"))


In [ ]:
# 18. Train regression models and compare performance

models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso Regression": make_pipeline(StandardScaler(), Lasso(alpha=0.05, max_iter=5000)),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
    ),
}

fitted_models = {}
predictions = {}
metric_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    fitted_models[name] = model
    predictions[name] = pred

    metric_rows.append({
        "model": name,
        "R2": r2_score(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    })

metrics_df = (
    pd.DataFrame(metric_rows)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(metrics_df.round(4))


In [ ]:
# 19. Model comparison dashboard

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metric_specs = [
    ("R2", "R2 Score", False),
    ("MAE", "Mean Absolute Error", True),
    ("RMSE", "Root Mean Squared Error", True),
]

for ax, (metric, ylabel, lower_is_better) in zip(axes, metric_specs):
    plot_df = metrics_df.sort_values(metric, ascending=lower_is_better)
    best_model = plot_df.iloc[0]["model"]
    colors = ["#2a9d8f" if model == best_model else "#8ecae6" for model in plot_df["model"]]

    bars = ax.bar(plot_df["model"], plot_df[metric], color=colors)
    label_offset = max(abs(plot_df[metric].max()), 1) * 0.015

    for bar, value in zip(bars, plot_df[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            f"{value:.4f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax.set_title(f"{metric}: {'lower is better' if lower_is_better else 'higher is better'}")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)

plt.suptitle("Regression Model Comparison", y=1.04)
plt.tight_layout()
plt.show()



In [ ]:
# 20. Actual vs predicted plots for all models

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for i, row in metrics_df.iterrows():
    name = row["model"]
    pred = predictions[name]
    ax = axes[i]

    ax.scatter(y_test, pred, alpha=0.15, s=10)
    limit_min = min(y_test.min(), pred.min())
    limit_max = max(y_test.max(), pred.max())
    ax.plot([limit_min, limit_max], [limit_min, limit_max], color="#e76f51", linestyle="--")
    ax.set_title(f"{name}\nR2={row['R2']:.4f}, RMSE={row['RMSE']:.2f}")
    ax.set_xlabel("Actual remaining atoms")
    ax.set_ylabel("Predicted remaining atoms")

plt.tight_layout()
plt.show()


In [ ]:
# 21. Residual analysis for the best model

best_model_name = metrics_df.loc[0, "model"]
best_pred = predictions[best_model_name]
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(best_pred, residuals, alpha=0.2, s=10)
axes[0].axhline(0, color="#e76f51", linestyle="--")
axes[0].set_title("Residuals vs Fitted")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

sns.histplot(residuals, bins=50, kde=True, ax=axes[1])
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")

stats.probplot(residuals.sample(min(2000, len(residuals)), random_state=RANDOM_STATE), dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot of Residuals")

plt.suptitle(f"Residual Diagnostics: {best_model_name}", y=1.04)
plt.tight_layout()
plt.show()

print(f"Best model: {best_model_name}")
print(f"Residual mean: {residuals.mean():.4f}")
print(f"Residual standard deviation: {residuals.std():.4f}")


In [ ]:
# 22. Feature importance from Random Forest

rf_model = fitted_models["Random Forest"]
rf_importance = (
    pd.Series(rf_model.feature_importances_, index=MODEL_FEATURES)
    .sort_values(ascending=True)
)

display(rf_importance.sort_values(ascending=False).to_frame("importance").round(4))

fig, ax = plt.subplots(figsize=(9, 6))
rf_importance.plot(kind="barh", ax=ax, color="#2a9d8f")
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# 23. Cross-validation on a representative training sample

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_X = X_train.sample(min(10000, len(X_train)), random_state=RANDOM_STATE)
cv_y = y_train.loc[cv_X.index]

cv_models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest": RandomForestRegressor(
        n_estimators=60,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
    ),
}

cv_rows = []
for name, model in cv_models.items():
    scores = cross_val_score(model, cv_X, cv_y, cv=cv, scoring="r2")
    cv_rows.append({
        "model": name,
        "mean_R2": scores.mean(),
        "std_R2": scores.std(),
        "fold_scores": ", ".join(f"{score:.4f}" for score in scores),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("mean_R2", ascending=False)
display(cv_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cv_df, x="model", y="mean_R2", ax=ax)
ax.errorbar(
    x=np.arange(len(cv_df)),
    y=cv_df["mean_R2"],
    yerr=cv_df["std_R2"],
    fmt="none",
    color="black",
    capsize=4,
)
ax.set_title("5-Fold Cross-Validation R2")
ax.set_xlabel("")
ax.set_ylabel("Mean R2")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 24. Final project summary

best_row = metrics_df.iloc[0]

print("=" * 70)
print("Quantum Radioactive Decay Analysis Summary")
print("=" * 70)
print(f"Dataset rows          : {len(df_feat):,}")
print(f"Dataset columns       : {df_feat.shape[1]}")
print(f"Missing after cleaning: {int(df_clean.isna().sum().sum())}")
print(f"Target variable       : {TARGET}")
print(f"Model feature count   : {len(MODEL_FEATURES)}")
print(f"Best model            : {best_row['model']}")
print(f"Best test R2          : {best_row['R2']:.4f}")
print(f"Best test MAE         : {best_row['MAE']:.4f}")
print(f"Best test RMSE        : {best_row['RMSE']:.4f}")
print("=" * 70)

print("Key findings:")
print("- Remaining atoms follows the expected exponential decay pattern.")
print("- Time, decay constant, half-life, and theoretical remaining atoms drive prediction quality.")
print("- Group labels add context, but the physics-based features explain most of the signal.")
print("- Model comparison uses R2, MAE, and RMSE so accuracy is judged from multiple angles.")
print("- Cross-validation checks whether the model performance is stable beyond one train/test split.")

